# 1. Setup

## 1.1 Imports

In [ ]:
import pandas as pd
import numpy as np

from pathlib import Path

import matplotlib.pyplot as plt

from numpy.linalg import lstsq
from scipy.stats import kruskal
import scikit_posthocs as sp
import statsmodels.formula.api as smf
import seaborn as sns

## 1.2 Paths

In [ ]:
input_path = Path("data/input/08_input")
output_path = Path("data/output/08_output")

## 1.3 Store TXT files as CSV

In [ ]:
for file_path in input_path.iterdir():
    # Skip directories, hidden/system files (e.g. .DS_Store), and any
    # non-data files. Adjust the suffix set to match your raw extensions.
    if not file_path.is_file():
        continue
    if file_path.name.startswith("."):
        continue
    if file_path.suffix.lower() not in {".txt", ".csv"}:
        continue

    file = pd.read_csv(file_path, sep="|")
    file.to_csv(file_path.with_suffix(".csv"), sep="|", index=False)

# 2. Clinical outcomes (predefined from V06)

In [ ]:
# Read file
all_clinical_08 = pd.read_csv(input_path / "AllClinical08.csv", sep="|")

## 2.1 Aggregate x-ray dataset fo KL grade

In [ ]:
"""
V08XRKL defines Kellgren and Lawrence grade from 0-4
0 = none (definite absence of x-ray changes of osteoarthritis)
1 = doubtful (doubtful joint space narrowing and possible osteophytic lipping)
2 = minimal (definite osteophytes and possible joint space narrowing)
3 = moderate (moderate multiple osteophytes, definite narrowing of joint space, some sclerosis and possible deformity of bone ends)
4 = severe (large osteophytes, marked narrowing of joint space, severe sclerosis and definite deformity of bone ends)
"""

xr_df = pd.read_csv(input_path / "KXR_SQ_BU08.csv", sep="|")

# select Kellgren and Lawrence Score and take max grade per ID and side
xr_df_grade = xr_df[["ID", "SIDE", "V08XRKL"]]
xr_df_grade = xr_df_grade.groupby(["ID", "SIDE"])["V08XRKL"].max().reset_index()

# clean label format from SIDE and V08XRKL (e.g. "1: Right" --> "Right", "2: 2" -> "2")
xr_df_grade["SIDE"] = xr_df_grade["SIDE"].str.extract(r":\s*(\w+)")
xr_df_grade["V08XRKL"] = xr_df_grade["V08XRKL"].str.extract(r":\s*(\d+)").squeeze().astype(float).astype("Int64")

# Pivot from long to wide format -> one row per ID, separate columns for Left and Right
xr_df_wide = xr_df_grade.pivot(index="ID", columns="SIDE", values="V08XRKL")
xr_df_wide.columns = [f"V08XRKL_{col}" for col in xr_df_wide.columns]
xr_df_wide = xr_df_wide.reset_index()

print(xr_df_wide)

## 2.2 Merge x-ray with clinical dataset

In [ ]:
all_clinical_08_merged = xr_df_wide.merge(all_clinical_08, on="ID", how="inner")

## 2.3 Predefined outcome variables (anchor_correlated_outcome_variables, V08 names)

In [ ]:
final_outcome_variables = [
    # pain_right — |rho| >= 0.20 threshold applied
    "V08KOOSKPR",  # KOOS Pain right knee — preferred over WOMAC (contains WOMAC, 0-100 scale)
    "V08ICPTSKR",  # ICOAP Right knee: Intermittent and Constant Pain Total Score, 0-100

    # pain_left — |rho| >= 0.20 threshold applied
    "V08KOOSKPL",  # KOOS Pain left knee
    "V08ICPTSKL",  # ICOAP Left knee: Intermittent and Constant Pain Total Score

    # function — |rho| >= 0.10 threshold applied
    "V0820MPACE",  # 20m walk pace (m/s) — represents full 20m walk cluster
    "V08CSTIME1",  # Repeated chair stand: time

    # self-reported function and symptoms right
    "V08WOMADLR",  # Right knee: WOMAC Disability Score (task-level self-reported function)
    "V08KOOSYMR",  # Right knee: KOOS Symptoms Score (distinct from pain and function)

    # self-reported function and symptoms left
    "V08WOMADLL",  # Left knee: WOMAC Disability Score (task-level self-reported function)
    "V08KOOSYML",  # Left knee: KOOS Symptoms Score (distinct from pain and function)

    # depression
    "V08CESD",     # CES-D total score

    # participation — |rho| >= 0.10 threshold applied
    "V08LLDILST",  # LLDI Limitation dimension total (perceived participation restriction)
    "V08LLDIFST",  # LLDI Frequency dimension total (frequency of participation)

    # quality of life
    "V08KOOSQOL",    # KOOS Quality of Life Score

]

In [ ]:
all_clinical_08_merged.to_csv(output_path / "all_clinical_08_merged.csv", sep="|", index=False)

# 3. Build cohort dataframes

## 3.1 Create Subject Summary Metrics dataframe

In [ ]:
# drop non-participants from the Accelerometry data for valid ID's
Accelerometry08 = pd.read_csv(input_path / "Accelerometry08.csv", sep="|")

accelerometry_valid_participants_08 = Accelerometry08[Accelerometry08["V08APASTAT"] != "Not participating"]

In [ ]:
# create summary_metrics_08 dataframe with relevant columns from accelerometry_valid_participants_08, all_clinical_08, and x-ray dataset (KL grade) and enrollees dataset

summary_metrics_08 = pd.DataFrame()

"""
Decision to use Trioano cut points for activity intensity classification, as these were validated in a population with rheumatic diseases and are commonly used in OAI accelerometer research. Freedson was validated on young healthy adults and underestimates MVPA in older populations, while Swartz overestimates it. The cut points are based on counts per minute (cpm) thresholds that correspond to different activity intensities:
light: 100-2019 cpm
moderate: 2020-5998 cpm
vigorous: >= 5999 cpm
"""

activity_cols = [
    "ID",
    "V08AACNT", # average daily counts
    "V08AALTMNT", # average daily light activity counts Trioano
    "V08AAMDMNT", # average daily moderate activity counts Trioano
    "V08AAMVMNT", # average daily moderate/vigorous activity counts Trioano
    "V08AAVMNT", # average daily vigorous activity counts Trioano
    "V08ANVDAYS", # number of valid days (exposed downstream as valid_days_oai)

    "V08AACSM03", # >= 30 minutes of moderate-intensity activity per day 0-1
    "V08ADHHS8", # >= 150 minutes of moderate activity and >=75 minutes of vigorous minutes per week 0 or 1
    "V08ADHHSD8", # >= 150 minutes of moderate-intensity activity per week 0 or 1
]

summary_metrics_08 = accelerometry_valid_participants_08[activity_cols].rename(
    columns={"V08ANVDAYS": "valid_days_oai"}
)

In [ ]:
print(f"Accelerometer participants: {Accelerometry08["ID"].nunique()}")
print(f"Accelerometer participants that participated: {accelerometry_valid_participants_08['ID'].nunique()}")
print(f"After merge: {summary_metrics_08['ID'].nunique()}")

## 3.2 Restrict V08 cohort to V06 analytic sample

In [ ]:
# Load the final V06 analytic cohort (post KL-grade, post-surgery filtering)
# to define eligibility for longitudinal validation.
summary_metrics_06 = pd.read_csv(Path("data/output/06_output") / "summary_metrics_06.csv",sep="|",)
v06_cohort_ids = set(summary_metrics_06["ID"])

In [ ]:
participants_before = summary_metrics_08["ID"].nunique()
summary_metrics_08 = summary_metrics_08[
    summary_metrics_08["ID"].isin(v06_cohort_ids)
].copy()
participants_after = summary_metrics_08["ID"].nunique()
print(f"V08 before V06 restriction: {participants_before:,}")
print(f"V08 after V06 restriction:  {participants_after:,}")
print(f"Dropped (not in V06 cohort): {participants_before - participants_after:,}")

In [ ]:
# merge outcome variables from all_clinical_08_merged
all_clinical_cols = [
    # basic parameter"ID",
    "ID", "V08AGE", "V08WEIGHT", "V08BMI", "V08COMORB", "V08CEMPLOY",
]
summary_metrics_08 = summary_metrics_08.merge(all_clinical_08_merged[all_clinical_cols], on="ID", how="left")
summary_metrics_08 = summary_metrics_08.merge(
    right=all_clinical_08_merged[["ID"] + final_outcome_variables],
    on="ID",
    how="left",
)

In [ ]:
print(summary_metrics_08.columns)

### Aggregate Enrollees for SEX column

In [ ]:
enrollees_df = pd.read_csv(input_path / "Enrollees.csv", sep="|")

#clean label format from P02SEX (e.g. "1: Male" --> "Male")
enrollees_df["P02SEX"] = enrollees_df["P02SEX"].str.extract(r":\s*(\w+)")


### Merge summary_metrics_08 with x-ray (KL grade) and enrollees (sex)

In [ ]:
summary_metrics_08 = (summary_metrics_08
                      .merge(xr_df_wide, on="ID", how="inner")
                      .merge(enrollees_df[["ID", "P02SEX"]], on="ID", how="inner")
                      )

# Verify — any drop beyond accelerometer filtering is a data quality signal
accelerometer_participant_count = accelerometry_valid_participants_08["ID"].nunique()
final_participant_count = summary_metrics_08["ID"].nunique()

# include KL grade per patient for later use in stratification and subgroup analyses; use worse knee (max of left and right) as KL grade per patient
kl_grade_per_patient = (
    xr_df_wide[["ID", "V08XRKL_Left", "V08XRKL_Right"]]
    .copy()
)
kl_grade_per_patient["kl_grade_index_knee"] = kl_grade_per_patient[
    ["V08XRKL_Left", "V08XRKL_Right"]
].max(axis=1)

# Merge KL grade into summary data
summary_metrics_08 = summary_metrics_08.merge(
    kl_grade_per_patient[["ID", "kl_grade_index_knee"]],
    on="ID",
    how="left",
)

print(f"Final participants after all merges: {final_participant_count}")

In [ ]:
restricted_ids = set(summary_metrics_06["ID"]) & set(accelerometry_valid_participants_08["ID"])
xr_ids  = set(xr_df_wide["ID"])
enr_ids = set(enrollees_df["ID"])

print(f"Cohort missing V08 x-ray:    {len(restricted_ids - xr_ids):,}")
print(f"Cohort missing enrollee row: {len(restricted_ids - enr_ids):,}")

In [ ]:
summary_metrics_08.to_csv(output_path / "summary_metrics_08.csv", sep="|", index=False)
print(summary_metrics_08.shape)

## 3.3 Drop participants with prior knee surgery (v07, v08)

In [ ]:
clinical_frames_by_visit = {
"v07": pd.read_csv(input_path / "AllClinical07.csv", sep="|"),
"v08": pd.read_csv(input_path / "AllClinical08.csv", sep="|"),
}

In [ ]:
ID_COLUMN: str = "ID"
YES_CODE: int = 1  # confirmed encoding: 1 = Yes (surgery), 0 = No

# Each visit's right/left "surgery or arthroscopy" variables. Baseline (P01)
# asks "ever"; each follow-up asks "since last visit ~12 months". The union
# across visits gives cumulative surgery history up to V08.
SURGERY_ITEMS_BY_VISIT: dict[str, tuple[str, str]] = {
    # visit_code: (right_knee_variable, left_knee_variable)
    "v07": ("V07KSRGR12", "V07KSRGL12"),
    "v08": ("V08KSRGR12", "V08KSRGL12"),
}

In [ ]:
def parse_surgery_code(raw_series: pd.Series) -> pd.Series:
    """Parse an OAI surgery item into a numeric code (0 = No, 1 = Yes).

    Handles both labelled values such as ``"1: Yes"`` and bare values such as
    ``1`` or ``"0"``. Any value whose leading token is not a digit (for example
    ``".: Missing Form/Incomplete Workbook"``) becomes missing.

    :param raw_series: Surgery item as read from the clinical file.
    :returns: Numeric series with missing values for non-coded entries.
    """
    leading_token = raw_series.astype(str).str.strip().str.split(":").str[0]
    return pd.to_numeric(leading_token, errors="coerce")

def merge_surgery_columns(
    clinical_frames_by_visit: dict[str, pd.DataFrame],
    *,
    surgery_items_by_visit: dict[str, tuple[str, str]] = SURGERY_ITEMS_BY_VISIT,
    id_column: str = ID_COLUMN,
) -> pd.DataFrame:
    """Merge the right/left surgery columns from every visit onto one frame.

    Only the identifier and the two surgery variables are taken from each
    visit, then merged on the identifier with an outer join so that no
    participant is dropped for being absent at a given visit. Surgery codes are
    coerced to numeric, turning blanks and stray codes into missing values.

    :param clinical_frames_by_visit: Mapping from visit label to its loaded
        data frame. Must contain every visit named in ``surgery_items_by_visit``.
    :param surgery_items_by_visit: Mapping from visit label to the right- and
        left-knee surgery variable names for that visit.
    :param id_column: Name of the participant identifier column.
    :returns: One row per participant, indexed by identifier, containing the
        surgery columns from all visits as numeric values.
    """
    merged_surgery_data: pd.DataFrame | None = None

    for visit_label, surgery_columns in surgery_items_by_visit.items():
        if visit_label not in clinical_frames_by_visit:
            raise KeyError(f"No data frame supplied for visit '{visit_label}'.")

        clinical_frame = clinical_frames_by_visit[visit_label]
        right_column, left_column = surgery_columns

        missing_columns = {id_column, right_column, left_column} - set(clinical_frame.columns)
        if missing_columns:
            raise KeyError(
                f"Visit '{visit_label}' is missing expected column(s): "
                f"{sorted(missing_columns)}"
            )

        visit_subset = clinical_frame[[id_column, right_column, left_column]].copy()
        visit_subset[[right_column, left_column]] = visit_subset[
            [right_column, left_column]
        ].apply(parse_surgery_code)

        if merged_surgery_data is None:
            merged_surgery_data = visit_subset
        else:
            merged_surgery_data = merged_surgery_data.merge(
                visit_subset, on=id_column, how="outer"
            )

    return merged_surgery_data.set_index(id_column)


def build_prior_knee_surgery_exclusion(
    merged_clinical_data: pd.DataFrame,
    *,
    surgery_items_by_visit: dict[str, tuple[str, str]] = SURGERY_ITEMS_BY_VISIT,
    yes_code: int = YES_CODE,
    treat_missing_as_surgery: bool = False,
) -> pd.Series:
    """Flag participants reporting any knee surgery across the included visits.

    A participant is flagged when any right- or left-knee surgery item, at any
    visit, equals ``yes_code``. Because the source encoding is 0 = No, 1 = Yes
    with "don't know" stored as missing, the missing-value policy is set
    explicitly rather than via a numeric code.

    :param merged_clinical_data: One row per participant, indexed by identifier,
        containing the surgery columns named in ``surgery_items_by_visit``.
    :param surgery_items_by_visit: Mapping from visit label to the right- and
        left-knee surgery variable names for that visit.
    :param yes_code: Encoded value representing an affirmative response.
    :param treat_missing_as_surgery: When ``True``, a participant whose every
        surgery item is missing is also flagged, for a conservative exclusion.
        When ``False`` (default), only explicit affirmative responses flag.
    :returns: Boolean series indexed by identifier, ``True`` where the
        participant should be excluded.
    """
    surgery_columns = [
        column
        for right_left_pair in surgery_items_by_visit.values()
        for column in right_left_pair
    ]

    affirmative = merged_clinical_data[surgery_columns].eq(yes_code)
    exclude = affirmative.any(axis="columns")

    if treat_missing_as_surgery:
        all_missing = merged_clinical_data[surgery_columns].isna().all(axis="columns")
        exclude |= all_missing

    return exclude


def summarize_surgery_by_visit(
    merged_clinical_data: pd.DataFrame,
    *,
    surgery_items_by_visit: dict[str, tuple[str, str]] = SURGERY_ITEMS_BY_VISIT,
    yes_code: int = YES_CODE,
) -> pd.DataFrame:
    """Count participants reporting knee surgery at each visit.

    For every visit, a participant counts as reporting surgery when either the
    right- or left-knee item equals ``yes_code``. Because baseline asks "ever"
    and follow-ups ask "since last visit", these counts are not mutually
    exclusive across visits and should not be summed into a total.

    :param merged_clinical_data: One row per participant, indexed by identifier,
        containing the surgery columns named in ``surgery_items_by_visit``.
    :param surgery_items_by_visit: Mapping from visit label to the right- and
        left-knee surgery variable names for that visit.
    :param yes_code: Encoded value representing an affirmative response.
    :returns: Data frame indexed by visit label, with the number reporting
        surgery, the number explicitly answering, and the number missing.
    """
    summary_rows: dict[str, dict[str, int]] = {}

    for visit_label, (right_column, left_column) in surgery_items_by_visit.items():
        visit_codes = merged_clinical_data[[right_column, left_column]]
        reported_surgery = visit_codes.eq(yes_code).any(axis="columns")
        answered = visit_codes.notna().any(axis="columns")

        summary_rows[visit_label] = {
            "reported_surgery": int(reported_surgery.sum()),
            "answered": int(answered.sum()),
            "missing_all": int((~answered).sum()),
        }

    return pd.DataFrame.from_dict(summary_rows, orient="index")


In [ ]:
merged_surgery_data = merge_surgery_columns(clinical_frames_by_visit)

# Sanity check: baseline 'answered' should be ~4788 and v01 ~4471, not 0.
surgery_by_visit = summarize_surgery_by_visit(merged_surgery_data)
print(surgery_by_visit)

surgery_exclusion_mask = build_prior_knee_surgery_exclusion(merged_surgery_data)
surgery_participant_ids = set(merged_surgery_data.index[surgery_exclusion_mask])

summary_metrics_08_filtered = summary_metrics_08.loc[
    ~summary_metrics_08[ID_COLUMN].isin(surgery_participant_ids)
].copy()

print(f"Participants flagged for knee surgery: {len(surgery_participant_ids)}")
print(f"Summary rows before: {len(summary_metrics_08)}")
print(f"Summary rows after:  {len(summary_metrics_08_filtered)}")

In [ ]:
def summarize_surgery_by_visit(
    merged_clinical_data: pd.DataFrame,
    *,
    surgery_items_by_visit: dict[str, tuple[str, str]] = SURGERY_ITEMS_BY_VISIT,
    yes_code: int = YES_CODE,
) -> pd.DataFrame:
    """Count participants reporting knee surgery at each visit.

    For every visit, a participant counts as reporting surgery when either the
    right- or left-knee item equals ``yes_code``. Because baseline asks "ever"
    and follow-ups ask "since last visit", these counts are not mutually
    exclusive across visits and should not be summed into a total.

    :param merged_clinical_data: One row per participant, indexed by identifier,
        containing the surgery columns named in ``surgery_items_by_visit``.
    :param surgery_items_by_visit: Mapping from visit label to the right- and
        left-knee surgery variable names for that visit.
    :param yes_code: Encoded value representing an affirmative response.
    :returns: Data frame indexed by visit label, with the number reporting
        surgery, the number explicitly answering, and the number missing.
    """
    summary_rows: dict[str, dict[str, int]] = {}

    for visit_label, (right_column, left_column) in surgery_items_by_visit.items():
        visit_codes = merged_clinical_data[[right_column, left_column]]
        reported_surgery = visit_codes.eq(yes_code).any(axis="columns")
        answered = visit_codes.notna().any(axis="columns")

        summary_rows[visit_label] = {
            "reported_surgery": int(reported_surgery.sum()),
            "answered": int(answered.sum()),
            "missing_all": int((~answered).sum()),
        }

    return pd.DataFrame.from_dict(summary_rows, orient="index")

In [ ]:
surgery_by_visit = summarize_surgery_by_visit(merged_surgery_data)
print(surgery_by_visit)

In [ ]:
# --- Exclude participants with any reported knee surgery

surgery_exclusion_mask = build_prior_knee_surgery_exclusion(merged_surgery_data)
surgery_participant_ids = set(merged_surgery_data.index[surgery_exclusion_mask])

# Guard: confirm IDs actually align before trusting the row count. A dtype or
# format mismatch makes .isin match nothing and silently drop zero rows.
summary_ids = set(summary_metrics_08["ID"])
overlap = surgery_participant_ids & summary_ids
print(f"Flagged for surgery (all visits): {len(surgery_participant_ids)}")
print(f"Of those, present in final summary cohort: {len(overlap)}")
assert overlap, "No flagged IDs matched summary_metrics_08 — check ID dtype/format."

before_surgery = summary_metrics_08["ID"].nunique()
summary_metrics_08 = summary_metrics_08[
    ~summary_metrics_08["ID"].isin(surgery_participant_ids)
].copy()
after_surgery = summary_metrics_08["ID"].nunique()

print(f"Dropped for prior/interval knee surgery: {before_surgery - after_surgery}")
print(f"Final participants (surgery-free): {after_surgery}")

In [ ]:
summary_metrics_08.to_csv(output_path / "summary_metrics_08.csv", sep="|", index=False)
print(summary_metrics_08.shape)

## 3.4 Create daily metrics

In [ ]:
pd.read_csv(input_path / "AccelDataByDay08.csv", sep="|")

daily_metrics_08 = pd.DataFrame()

# define columns from AccelDataByDay08
cols = [
    "ID",
    "V08PAWeekDay",
    "V08PAStudyDay",
    "V08DAYCnt", # total counts per day
    "V08DAYLtMinT", # minutes of light activity (Troiano)
    "V08DAYModMinT", # minutes of moderate activity (Troiano)
    "V08DAYVigMinT", # minutes of vigorous activity (Troiano)
    "V08DAYMVMinT", # minutes of moderate to vigorous activity (Troiano)
    "V08WearHr", # wear time in minutes
    ]

daily_metrics_08 = pd.read_csv(input_path / "AccelDataByDay08.csv", sep="|")[cols]
daily_metrics_08 = daily_metrics_08.rename(columns={"V08PAWeekDay": "week_day"})

# Merge KL grade into daily data
daily_metrics_08 = daily_metrics_08.merge(
    kl_grade_per_patient[["ID", "kl_grade_index_knee"]],
    on="ID",
    how="left",
)

## 3.5 Create minute metrics

In [ ]:
Acceldatabymin08 = pd.read_csv(input_path / "Acceldatabymin08.csv", sep="|")

# Create minute metrics dataframe
pd.read_csv(input_path / "Acceldatabymin08.csv", sep="|")

minute_metrics_08 = pd.DataFrame()

# define columns from Acceldatabymin08
cols = [
    "ID",
    "V08PAStudyDay",
    "V08PAWeekDay",
    "V08MinSequence",
    "V08MINCnt",
    "V08SuspectMinute"
]

minute_metrics_08 = Acceldatabymin08[cols]

In [ ]:
def compare_dataframe_ids(
    *,
    first_dataframe: pd.DataFrame,
    second_dataframe: pd.DataFrame,
    id_column: str = "id",
) -> dict[str, set]:
    """Compare ID values between two dataframes.

    :param first_dataframe: The first dataframe to compare.
    :param second_dataframe: The second dataframe to compare against.
    :param id_column: Name of the column containing IDs in both dataframes.
    :returns: Dictionary with sets of IDs that are common, only in the first,
        and only in the second dataframe.
    """
    first_identifiers = set(first_dataframe[id_column])
    second_identifiers = set(second_dataframe[id_column])

    return {
        "in_both": first_identifiers & second_identifiers,
        "only_in_first": first_identifiers - second_identifiers,
        "only_in_second": second_identifiers - first_identifiers,
    }

In [ ]:
# compair daily_metrics and minute_metrics ID's
comparison_result = compare_dataframe_ids(
    first_dataframe=accelerometry_valid_participants_08,
    second_dataframe= Acceldatabymin08,
    id_column="ID",
)
print(f"Shared IDs: {len(comparison_result['in_both'])}")
print(f"Only in first: {len(comparison_result['only_in_first'])}")
print(f"Only in second: {len(comparison_result['only_in_second'])}")

In [ ]:
minute_metrics_08 = minute_metrics_08.merge(
    kl_grade_per_patient[["ID", "kl_grade_index_knee"]],
    on="ID",
    how="left",
)

In [ ]:
# Restrict accelerometry detail frames to the final summary cohort.
# summary_metrics_06 is already filtered for usable KL grade and
# prior/interval knee surgery, so its IDs define the analytic sample.
final_participant_ids = set(summary_metrics_08["ID"])

minute_metrics_08 = minute_metrics_08[
    minute_metrics_08["ID"].isin(final_participant_ids)
].copy()

In [ ]:
# compair daily_metrics and minute_metrics ID's
comparison_result = compare_dataframe_ids(
    first_dataframe=summary_metrics_08,
    second_dataframe=minute_metrics_08,
    id_column="ID",
)
print(f"Shared IDs: {len(comparison_result['in_both'])}")
print(f"Only in first: {len(comparison_result['only_in_first'])}")
print(f"Only in second: {len(comparison_result['only_in_second'])}")

In [ ]:
minute_metrics_08.to_csv(output_path / "minute_metrics_08.csv", sep="|", index=False)

In [ ]:
print(minute_metrics_08["ID"].nunique())

# 4. Minute-level preparation & cleaning

## 4.1 Intensity classification (non-wear, inentsity labels)

In [ ]:
def identify_non_wear_minutes(
        dataframe: pd.DataFrame,
        non_wear_threshold_minutes: int = 90,
) -> pd.Series:

    """
    Identify non-wear minutes using a rolling window of consecutive zero
    activity counts per participant and study day.
    Non-wear periods are identified using the OAI-specific threshold of 90
    consecutive minutes of zero activity counts, which was validated for
    rheumatic disease populations.
    """

    is_non_wear = pd.Series(False, index=dataframe.index)

    for(id, study_day), group in dataframe.groupby(["ID", "study_day"]):

        zero_counts = group["counts"] == 0
        consecutive_zero_count = 0
        group_non_wear = pd.Series(False, index=group.index)
        bout_start_index = None

        for index, is_zero in zero_counts.items():
            if is_zero:
                if consecutive_zero_count == 0:
                    bout_start_index = index
                consecutive_zero_count += 1
            else:
                if consecutive_zero_count >= non_wear_threshold_minutes:
                    group_non_wear.loc[bout_start_index:index - 1] = True
                consecutive_zero_count = 0
                bout_start_index = None

        if consecutive_zero_count >= non_wear_threshold_minutes:
            group_non_wear.loc[bout_start_index:] = True


        is_non_wear.loc[group.index] = group_non_wear

    return is_non_wear

def assign_intensity_labels(dataframe: pd.DataFrame) -> pd.Series:

    """
    Assign an intensity label to each minute based on activity counts,
    non-wear status, and suspicious minute flag.

    Labels are assigned in the following priority order (Troiano):
        1. suspicious  — is_suspicious == True
        2. non_wear    — within a 90-minute consecutive zero-count period
        3. sedentary   — 0–99 counts/min (valid wear time)
        4. light       — 100–2019 counts/min
        5. moderate    — 2020–5998 counts/min
        6. vigorous    — >= 5999 counts/min
    """

    conditions = [
        dataframe["is_suspicious"],
        dataframe["is_non_wear"],
        dataframe["counts"] < 100,
        dataframe["counts"] < 2020,
        dataframe["counts"] < 5999,
        ]

    intensity_labels = [
        "suspicious",
        "non_wear",
        "sedentary",
        "light",
        "moderate",
    ]

    return pd.Series(
        np.select(
            condlist=conditions,
            choicelist=intensity_labels,
            default="vigorous",
        ),
        index=dataframe.index,
    )

def classify_activity_level(
        minute_dataframe: pd.DataFrame,
        non_wear_threshold_minutes: int = 90,
) -> pd.DataFrame:

    """
    Classify each minute of accelerometer data into an intensity label and
    add non-wear and suspicious minute flags to the dataframe.
    """

    required_columns = [
        "ID",
        "V08PAStudyDay",
        "V08PAWeekDay",
        "V08MinSequence",
        "V08MINCnt",
        "V08SuspectMinute",
    ]

    missing_columns = [
        column for column in required_columns
        if column not in minute_dataframe.columns
    ]

    if missing_columns:
        raise KeyError(
            f"The following required columns are missing: {missing_columns}"
        )

    result_dataframe = minute_dataframe.copy()

    result_dataframe = result_dataframe.rename(
        columns={
            "V08PAStudyDay": "study_day",
            "V08PAWeekDay": "week_day",
            "V08MinSequence": "minute_sequence",
            "V08MINCnt": "counts",
            "V08SuspectMinute": "is_suspicious",
        }
    )

    result_dataframe["is_suspicious"] = (result_dataframe["is_suspicious"] == 1)

    result_dataframe["is_non_wear"] = identify_non_wear_minutes(
        dataframe=result_dataframe,
        non_wear_threshold_minutes=non_wear_threshold_minutes,
    )

    result_dataframe["intensity_label"] = assign_intensity_labels(
        dataframe=result_dataframe,
    )

    return result_dataframe


In [ ]:
minute_metrics_08 = classify_activity_level(minute_dataframe=minute_metrics_08,)

In [ ]:
minute_metrics_08.to_csv(output_path / "minute_metrics_08.csv", sep="|", index=False)

## 4.2 Remove fully non-wear days

In [ ]:
# Identify fully non-wear days
fully_non_wear_days = (
    minute_metrics_08
    .groupby(["ID", "study_day"])
    .apply(lambda day: (day["is_non_wear"] == True).all())
)

fully_non_wear_days = fully_non_wear_days[fully_non_wear_days]
print(f"Participant-days where entire day is non-wear: {len(fully_non_wear_days)}")
print(f"Participants affected: {fully_non_wear_days.index.get_level_values('ID').nunique()}")


In [ ]:
# Drop fully non wear days from the minute-level dataset to avoid skewing the harmonic regression
fully_non_wear_days = fully_non_wear_days[fully_non_wear_days]

minute_metrics_08 = minute_metrics_08[
    ~minute_metrics_08.set_index(["ID", "study_day"]).index.isin(fully_non_wear_days.index)
].reset_index(drop=True)

In [ ]:
print(f"Participants before non-wear days drop {minute_metrics_08["ID"].nunique()}")

In [ ]:
print(f"Participants after non-wear days drop {minute_metrics_08["ID"].nunique()}")

In [ ]:
# non wear distribution plot after cleaning
non_wear_per_minute = (
    minute_metrics_08[minute_metrics_08["is_non_wear"] == True]
    .groupby("minute_sequence")["ID"]
    .nunique()
)

plt.figure(figsize=(12, 4))
plt.bar(non_wear_per_minute.index, non_wear_per_minute.values, width=1)
plt.xlabel("Minute sequence")
plt.ylabel("Number of participants")
plt.title("Number of participants with non-wear at each minute position (after cleaning)")
plt.xticks(
    ticks=[0, 180, 360, 540, 720, 900, 1080, 1260, 1440],
    labels=["00:00", "03:00", "06:00", "09:00", "12:00", "15:00", "18:00", "21:00", "24:00"],
)
plt.show()

## 4.3 Remove suspicious participant-days

In [ ]:
# Explore suspicious minutes distribution across the day

suspicious_minutes = (
    minute_metrics_08[
        minute_metrics_08["is_suspicious"] == True
    ]["minute_sequence"]
)

plt.figure(figsize=(12, 4))
plt.hist(suspicious_minutes, bins=100)
plt.xlabel("Minute sequence")
plt.ylabel("Count")
plt.title("Distribution of suspicious minutes across the day")
plt.xticks(
    ticks=[0, 180, 360, 540, 720, 900, 1080, 1260, 1440],
    labels=["00:00", "03:00", "06:00", "09:00", "12:00", "15:00", "18:00", "21:00", "24:00"],
)
plt.show()

In [ ]:
# Drop only participant-days that contain any suspect minute, preserving each
# participant's remaining clean days. A single flagged minute is treated as
# compromising that day's rhythm fit, not the participant's entire record.

day_columns = ["ID", "study_day"]

suspect_participant_days = (
    minute_metrics_08.loc[minute_metrics_08["is_suspicious"] == True, day_columns]
    .drop_duplicates()
)

minutes_before = len(minute_metrics_08)
participants_before = minute_metrics_08["ID"].nunique()

# Anti-join: keep rows whose (ID, study day) is NOT in the suspect-day set.
minute_metrics_08 = (
    minute_metrics_08.merge(
        suspect_participant_days,
        on=day_columns,
        how="left",
        indicator=True,
    )
    .query("_merge == 'left_only'")
    .drop(columns="_merge")
    .copy()
)

print(f"Suspect participant-days removed: {len(suspect_participant_days):,}")
print(f"Minutes dropped: {minutes_before - len(minute_metrics_08):,}")
print(f"Participants before: {participants_before:,}")
print(f"Participants after:  {minute_metrics_08['ID'].nunique():,}")

## 4.4 Remove days with implausible wear time
Drop days with less than 10 hours of wear wear time to ensure stable harmonic regression fits

In [ ]:
wear_time_per_day = (
    minute_metrics_08
    .groupby(["ID", "study_day"])["is_non_wear"]
    .apply(lambda x: (x == False).sum() / 60)
    .reset_index()
    .rename(columns={"is_non_wear": "wear_hours"})
)

invalid_days = wear_time_per_day[
    (wear_time_per_day["wear_hours"] < 10)
][["ID", "study_day"]]

minute_metrics_08 = minute_metrics_08[
    ~minute_metrics_08.set_index(["ID", "study_day"]).index.isin(
        invalid_days.set_index(["ID", "study_day"]).index
    )
].reset_index(drop=True)

print(f"Dropped {len(invalid_days):,} days.")
print(f"Remaining participants: {minute_metrics_08['ID'].nunique():,}")

## 4.5 Remove participants with <4 valid days

In [ ]:
days_per_participant = (
    minute_metrics_08
    .groupby("ID")["study_day"]
    .nunique()
)

valid_participants = days_per_participant[days_per_participant >= 4].index

participants_before = minute_metrics_08["ID"].nunique()
minute_metrics_08 = minute_metrics_08[minute_metrics_08["ID"].isin(valid_participants)].reset_index(drop=True)
participants_after = minute_metrics_08["ID"].nunique()

print(f"Dropped {participants_before - participants_after:,} participants with fewer than 4 valid days.")
print(f"Remaining participants: {participants_after:,}")

### Compare IDs between summary metric and minute metric datasets to check for any discrepancies after cleaning

In [ ]:
# compair daily_metrics and minute_metrics ID's
comparison_result = compare_dataframe_ids(
    first_dataframe=daily_metrics_08,
    second_dataframe=minute_metrics_08,
    id_column="ID",
)
print(f"Shared IDs: {len(comparison_result['in_both'])}")
print(f"Only in first: {len(comparison_result['only_in_first'])}")
print(f"Only in second: {len(comparison_result['only_in_second'])}")

### 4.6 Synchronize valid IDs across all dataframes

In [ ]:
valid_ids_after_cleaning = set(minute_metrics_08["ID"].unique())

daily_metrics_08 = daily_metrics_08[
    daily_metrics_08["ID"].isin(valid_ids_after_cleaning)
].reset_index(drop=True)

# summary_metrics_08 uses ID as index at this point in the pipeline
summary_metrics_08 = summary_metrics_08[
    summary_metrics_08["ID"].isin(valid_ids_after_cleaning)
].reset_index(drop=True)

print(
    f"Participants after synchronisation:\n"
    f"  minute_metrics_08 : {minute_metrics_08['ID'].nunique():,}\n"
    f"  daily_metrics_08  : {daily_metrics_08['ID'].nunique():,}\n"
    f"  summary_metrics_08: {len(summary_metrics_08):,}"
)

# 5. Activity feature engineering

## 5.1 Bout structure

In [ ]:
def calculate_bout_structure(
        minute_dataframe: pd.DataFrame,
) -> pd.DataFrame:
    """
    Calculate bout structure parameters for each participant and study day
    from minute-level accelerometer data.

    For each intensity level, the following parameters are calculated:
        - bout_count:          number of uninterrupted episodes per day
        - bout_mean_duration:  mean duration of episodes in minutes
        - bout_max_duration:   longest episode in minutes
        - bout_total_minutes:  total minutes accumulated in episodes

    Intensity levels calculated:
        - sedentary:  < 100 counts/min
        - light:      100–2019 counts/min
        - moderate:   2020–5998 counts/min
        - vigorous:   >= 5999 counts/min
        - mvpa:       moderate + vigorous (>= 2020 counts/min)
        - active:     light + moderate + vigorous (>= 100 counts/min)

    Non-wear and suspicious minutes are excluded before calculation.
    A bout is defined as consecutive minutes of the same intensity level
    with no tolerance for interruptions.
    """

    required_columns =[
        "ID",
        "study_day",
        "counts",
        "minute_sequence",
        "is_non_wear",
        "is_suspicious",
    ]

    missing_columns = [
        column for column in required_columns
        if column not in minute_dataframe.columns
    ]

    if missing_columns:
        raise KeyError(
            f"The following required columns are missing: {missing_columns}"
        )

    intensity_level_definition = {
        "sedentary": lambda counts: counts < 100,
        "light": lambda counts: (counts >= 100) & (counts < 2020),
        "moderate": lambda counts: (counts >= 2020) & (counts < 5999),
        "vigorous": lambda counts: counts >= 5999,
        "mvpa": lambda counts: counts >= 2020,
        "active": lambda counts: counts >= 100,
    }

    valid_minutes = minute_dataframe[
        ~minute_dataframe["is_non_wear"]
        & ~minute_dataframe["is_suspicious"]
    ].copy()

    valid_minutes = valid_minutes.dropna(subset=["counts"])

    results = []

    for (participant_id, study_day), group in valid_minutes.groupby(["ID", "study_day"]
    ):
        day_result = {
            "ID": participant_id,
            "study_day": study_day,
        }

        for intensity_level, intensity_condition in intensity_level_definition.items():
            bout_duration = extract_bout_duration(
                counts=group["counts"],
                minute_sequence=group["minute_sequence"],
                intensity_condition=intensity_condition,
            )

            if len(bout_duration) == 0:
                day_result[f"{intensity_level}_bout_count"] = 0
                day_result[f"{intensity_level}_bout_mean_duration"] = 0.0
                day_result[f"{intensity_level}_bout_max_duration"] = 0.0
                day_result[f"{intensity_level}_bout_total_minutes"] = 0.0
            else:
                day_result[f"{intensity_level}_bout_count"] = len(bout_duration)
                day_result[f"{intensity_level}_bout_mean_duration"] = np.mean(bout_duration)
                day_result[f"{intensity_level}_bout_max_duration"] = np.max(bout_duration)
                day_result[f"{intensity_level}_bout_total_minutes"] = np.sum(bout_duration)

        results.append(day_result)

    return pd.DataFrame(results)

def extract_bout_duration(
        counts: pd.Series,
        minute_sequence: pd.Series,
        intensity_condition: callable,
        expected_minute_step: int = 1,
) -> list[int]:

    """
    Extract the duration of each uninterrupted bout matching the given
    intensity condition from a minute-level counts series.

    A bout is a sequence of consecutive minutes for which the intensity
    condition is met *and* which are adjacent in time. A bout is broken
    whenever the intensity condition fails, or whenever the gap between
    two successive retained minutes exceeds ``expected_minute_step`` (for
    example because a non-wear, suspicious, or missing minute was removed
    upstream).

    :param counts: Minute-level activity counts.
    :param minute_sequence: Within-day minute position for each minute,
        index-aligned with ``counts``.
    :param intensity_condition: Callable mapping a counts series to a
        boolean series that is ``True`` where the minute matches the
        target intensity.
    :param expected_minute_step: Maximum allowed difference in
        ``minute_sequence`` between two successive minutes for them to
        count as adjacent. Defaults to one minute.
    :returns: Duration in minutes of each bout, in order of occurrence.
    """

    minute_frame = pd.DataFrame({
        "minute_sequence": minute_sequence,
        "is_intensity": intensity_condition(counts),
    }).sort_values("minute_sequence").reset_index(drop=True)

    bout_durations = []
    current_bout_duration = 0
    previous_minute_sequence = None

    for minute_position, is_active in zip(
            minute_frame["minute_sequence"], minute_frame["is_intensity"]
    ):
        is_adjacent = (
            previous_minute_sequence is not None
            and (minute_position - previous_minute_sequence) <= expected_minute_step
        )

        # A time gap ends the current bout even if this minute also
        # matches the intensity condition.
        if is_active and current_bout_duration > 0 and not is_adjacent:
            bout_durations.append(current_bout_duration)
            current_bout_duration = 0

        if is_active:
            current_bout_duration += 1
        else:
            if current_bout_duration > 0:
                bout_durations.append(current_bout_duration)
            current_bout_duration = 0

        previous_minute_sequence = minute_position

    if current_bout_duration > 0:
        bout_durations.append(current_bout_duration)

    return bout_durations

In [ ]:
# calculate bout structure

bout_structure_daily = calculate_bout_structure(minute_dataframe=minute_metrics_08)

# merge into daily_metrics_08
daily_metrics_08 = daily_metrics_08.merge(
    bout_structure_daily,
    left_on=["ID", "V08PAStudyDay"],
    right_on=["ID", "study_day"],
    how="left",
)

# aggregate to summary level
bout_structure_summary = (
    bout_structure_daily
    .groupby("ID")
    .agg({
        col: "mean"
        for col in bout_structure_daily.columns
        if col not in ["ID", "study_day"]
    })
    .reset_index()
    .rename(columns={
        col: f"mean_{col}"
        for col in bout_structure_daily.columns
        if col not in ["ID", "study_day"]
    })
)

# merge into summary_metrics_08
summary_metrics_08 = summary_metrics_08.merge(
    bout_structure_summary,
    on="ID",
    how="left",
)

In [ ]:
summary_metrics_08.to_csv(output_path / "summary_metrics_08.csv", sep="|", index=False)
daily_metrics_08.to_csv(output_path / "daily_metrics_08.csv", sep="|", index=False)

## 5.2 WHO guideline compliance

In [ ]:
def calculate_who_guideline_compliance(daily_dataframe: pd.DataFrame) -> pd.DataFrame:
    """
    Calculate WHO guideline compliance for each day based on daily activity metrics.
    WHO guidelines for adults aged 18-64 recommend:
        - At least 150 minutes of moderate-intensity aerobic physical activity per week, or
        - At least 75 minutes of vigorous-intensity aerobic physical activity per week, or
        - An equivalent combination of moderate- and vigorous-intensity activity.

    This function adds a column to the daily_metrics_df indicating whether the
    participant met the WHO guidelines on that day.
    """

    required_columns = [
        "ID",
        "V08PAStudyDay",
        "V08DAYModMinT",
        "V08DAYVigMinT",
    ]

    missing_columns = [
        column for column in required_columns
        if column not in daily_dataframe.columns
    ]
    if missing_columns:
        raise KeyError(
            f"The following required columns are missing: {missing_columns}"
        )

    daily_moderate_guideline_threshold = 150/7
    daily_vigorous_guideline_threshold = 75/7
    weekly_moderate_guideline_threshold = 150
    weekly_vigorous_guideline_threshold = 75
    vigorous_to_moderate_multiplier = 2

    result_dataframe = daily_dataframe.copy()

    result_dataframe["combined_equivalent_minutes"] = (result_dataframe["V08DAYModMinT"] + result_dataframe["V08DAYVigMinT"] * vigorous_to_moderate_multiplier)

    result_dataframe["meets_daily_WHO_guideline"] = (
    (result_dataframe["V08DAYModMinT"] >= daily_moderate_guideline_threshold)
    | (result_dataframe["V08DAYVigMinT"] >= daily_vigorous_guideline_threshold)
    | (result_dataframe["combined_equivalent_minutes"] >= daily_moderate_guideline_threshold)
    )

    weekly_compliance = result_dataframe.groupby("ID").agg(
        total_moderate_minutes=("V08DAYModMinT", "sum"),
        total_vigorous_minutes=("V08DAYVigMinT", "sum"),
        total_combined_equivalent_minutes=("combined_equivalent_minutes", "sum"),
        day_count=("combined_equivalent_minutes", "count"),
    ).reset_index()

    weekly_compliance["meets_weekly_who_guideline"] = (
    (weekly_compliance["total_moderate_minutes"] / weekly_compliance["day_count"] * 7 >= weekly_moderate_guideline_threshold)
    | (weekly_compliance["total_vigorous_minutes"] / weekly_compliance["day_count"] * 7 >= weekly_vigorous_guideline_threshold)
    | (weekly_compliance["total_combined_equivalent_minutes"] / weekly_compliance["day_count"] * 7 >= weekly_moderate_guideline_threshold)
)

    weekly_compliance["weekly_guideline_gap_minutes"] = (
    weekly_compliance["total_combined_equivalent_minutes"] / weekly_compliance["day_count"] * 7
    - weekly_moderate_guideline_threshold
)
    result_dataframe = result_dataframe.merge(
        weekly_compliance[[
            "ID",
            "meets_weekly_who_guideline",
            "weekly_guideline_gap_minutes",
        ]],
        on="ID",
        how="left",
    )

    return result_dataframe

In [ ]:
daily_metrics_08 = calculate_who_guideline_compliance(
    daily_dataframe=daily_metrics_08,
)

In [ ]:
summary_metrics_08 = summary_metrics_08.merge(
    daily_metrics_08[["ID", "meets_weekly_who_guideline", "weekly_guideline_gap_minutes"]]
    .drop_duplicates(subset="ID"),
    on="ID",
    how="left",
)

In [ ]:
summary_metrics_08["comparison"] = np.where(
    summary_metrics_08["V08ADHHS8"].isna()
    | summary_metrics_08["meets_weekly_who_guideline"].isna(),
    np.nan,
    (
        ((summary_metrics_08["V08ADHHS8"] == 1) & (summary_metrics_08["meets_weekly_who_guideline"] == True))
        | ((summary_metrics_08["V08ADHHS8"] == 0) & (summary_metrics_08["meets_weekly_who_guideline"] == False))

    ).map({True: "consistent", False: "inconsistent"}))

In [ ]:
summary_metrics_08.to_csv(output_path / "summary_metrics_08.csv", sep="|", index=False)

In [ ]:
participant_compliance = daily_metrics_08.groupby("ID")["meets_weekly_who_guideline"].first()

total = len(participant_compliance)
meets_true = participant_compliance.sum()
meets_false = total - meets_true

print(f"Total participants: {total}")
print(f"Meets WHO guideline: {meets_true} ({meets_true/total:.1%})")
print(f"Does not meet WHO guideline: {meets_false} ({meets_false/total:.1%})")

## 5.3 Activity onset / offset

In [ ]:
def compute_activity_onset_offset_time(df: pd.DataFrame, id_col: str = "ID", day_col: str = "study_day",
                                       min_col: str = "minute_sequence", activity_count_col: str = "counts",
                                       suspect_col: str = "is_suspicious") -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    computes activity onset (first active minute) and activity offset (last active minute) per ID and day
    Returns:
        daily_activity_onset_offset_metrics: ID, Day, activity_onset_minute, activity_offset_minute, onset_time_hhmm, offset_time_hhmm
        id_agg_sctivity_onset_offset_metrics: ID, aggregated onset/offset mean and sds
    """

# function to convert minute of day to HH:MM format, minute 1 = 00:01, minute 1440 = 24:00

    def minute_to_hhmm(m):
        if pd.isna(m):
            return np.nan
        h = int((m - 1) // 60)
        mins = int((m - 1) % 60)
        return f"{h:02d}:{mins:02d}"

    results = []

# filter per ID and day, only non-suspect minutes, sort by minute sequence, find first and last active minute (activity count > 0)

    for (id, day), grp in df.groupby([id_col, day_col]):

        grp_filtered = grp[grp[suspect_col] == 0].sort_values(min_col)
        active = grp_filtered[grp_filtered[activity_count_col] > 0][min_col].values

        if len(active) == 0:
            results.append({
                id_col: id,
                day_col: day,
                "activity_onset_minute": np.nan,
                "activity_offset_minute": np.nan,
            })
            continue

        results.append({
            id_col: id,
            day_col: day,
            "activity_onset_minute": active[0],  # first active minute
            "activity_offset_minute": active[-1],  # last active minute
        })

    daily_activity_onset_offset_metrics = pd.DataFrame(results)

    daily_activity_onset_offset_metrics["onset_time_hhmm"] = daily_activity_onset_offset_metrics["activity_onset_minute"].apply(minute_to_hhmm)
    daily_activity_onset_offset_metrics["offset_time_hhmm"] = daily_activity_onset_offset_metrics["activity_offset_minute"].apply(minute_to_hhmm)
    daily_activity_onset_offset_metrics["wear_duration_min"] = daily_activity_onset_offset_metrics["activity_offset_minute"] - daily_activity_onset_offset_metrics["activity_onset_minute"]

    # aggregate per ID across days

    id_agg_activity_onset_offset_metrics = (
        daily_activity_onset_offset_metrics.groupby(id_col).agg(
            activity_onset_minute_mean=("activity_onset_minute", np.mean),
            activity_onset_minute_sd=("activity_onset_minute", np.std),
            activity_offset_minute_mean=("activity_offset_minute", np.mean),
            activity_offset_minute_sd=("activity_offset_minute", np.std),
            wear_duration_mean=("wear_duration_min", np.mean),
            valid_days_derived=("activity_onset_minute", "count"),
        )
        .reset_index()
    )

    id_agg_activity_onset_offset_metrics["onset_mean_hhmm"] = id_agg_activity_onset_offset_metrics["activity_onset_minute_mean"].apply(minute_to_hhmm)
    id_agg_activity_onset_offset_metrics["offset_mean_hhmm"] = id_agg_activity_onset_offset_metrics["activity_offset_minute_mean"].apply(minute_to_hhmm)

    return daily_activity_onset_offset_metrics, id_agg_activity_onset_offset_metrics

In [ ]:
daily_activity_onset_offset_metrics_08, id_agg_activity_onset_offset_metrics_08 = compute_activity_onset_offset_time(minute_metrics_08)
summary_metrics_08 = summary_metrics_08.merge(id_agg_activity_onset_offset_metrics_08, on="ID", how="left")
daily_metrics_08 = daily_metrics_08.merge(daily_activity_onset_offset_metrics_08, on=["ID", "study_day"], how="left")

In [ ]:
summary_metrics_08.to_csv(output_path / "summary_metrics_08.csv", sep="|", index=False)
daily_metrics_08.to_csv(output_path / "daily_metrics_08.csv", sep="|", index=False)

## 5.4 Valid-day discrepancy exploration

In [ ]:
discrepant = summary_metrics_08[
    summary_metrics_08["valid_days_derived"] != summary_metrics_08["valid_days_oai"]
][["ID", "valid_days_derived", "valid_days_oai", "wear_duration_mean"]].dropna()

print(f"Number of discrepant participants: {len(discrepant)}")
print(discrepant.sort_values("ID"))
print(discrepant.shape)

# separately check how many have mean wear duration under 600 minutes (10 hours)
under_600 = discrepant[discrepant["wear_duration_mean"] < 600]
print(f"\nParticipants with mean wear duration under 600 minutes: {len(under_600)}")
print(under_600.sort_values("wear_duration_mean"))
print(under_600.shape)

# check suspicious minutes for discrepant participants
discrepant_ids = discrepant["ID"].tolist()

suspicious_summary = (
    minute_metrics_08[minute_metrics_08["ID"].isin(discrepant_ids)]
    .groupby("ID")["is_suspicious"]
    .agg(
        total_minutes="count",
        suspicious_minutes="sum",
    )
    .assign(
        suspicious_percent=lambda x: (x["suspicious_minutes"] / x["total_minutes"] * 100).round(1)
    )
    .reset_index()
)

discrepant = discrepant.merge(suspicious_summary, on="ID", how="left")

print(f"\nDiscrepant participants with suspicious minutes:")
print(discrepant[discrepant["suspicious_minutes"] > 0].sort_values("suspicious_minutes", ascending=False))
print(discrepant[discrepant["suspicious_minutes"] > 0].sort_values("suspicious_minutes", ascending=False).shape)
print(f"\nDiscrepant participants without suspicious minutes:")
print(discrepant[discrepant["suspicious_minutes"] == 0].sort_values("ID"))
print(discrepant[discrepant["suspicious_minutes"] == 0].sort_values("ID").shape)

# participants that are both suspicious AND under 600 minutes wear duration
print(f"\nDiscrepant participants with suspicious minutes AND under 600 minutes wear duration:")
suspicious_and_under_600 = discrepant[
    (discrepant["suspicious_minutes"] > 0)
    & (discrepant["wear_duration_mean"] < 600)
]
print(suspicious_and_under_600.sort_values("wear_duration_mean"))
print(suspicious_and_under_600.shape)


In [ ]:
print(summary_metrics_08[summary_metrics_08["valid_days_derived"] > 7].shape)

In [ ]:
# count all days per participant from minute data
all_days_from_minutes = (
    minute_metrics_08
    .groupby("ID")["study_day"]
    .nunique()
    .reset_index()
    .rename(columns={"study_day": "total_days_minute_data"})
)

# count valid days per participant from minute data
# a day is valid if it has at least one non-suspicious, non-wear minute
valid_days_from_minutes = (
    minute_metrics_08[
        ~minute_metrics_08["intensity_label"].isin(["non_wear", "suspicious"])
    ]
    .groupby("ID")["study_day"]
    .nunique()
    .reset_index()
    .rename(columns={"study_day": "valid_days_minute_data"})
)

# count days per participant from daily data (OAI valid days)
oai_valid_days = (
    daily_metrics_08
    .groupby("ID")["V08PAStudyDay"]
    .nunique()
    .reset_index()
    .rename(columns={"V08PAStudyDay": "oai_valid_days"})
)

# merge all three together with valid_days_oai from summary
day_comparison = (
    summary_metrics_08[["ID", "valid_days_oai"]]
    .merge(all_days_from_minutes, on="ID", how="left")
    .merge(valid_days_from_minutes, on="ID", how="left")
    .merge(oai_valid_days, on="ID", how="left")
)

# add difference columns to understand exclusion reasons
day_comparison["days_excluded_total"] = (
    day_comparison["total_days_minute_data"] - day_comparison["oai_valid_days"]
)
day_comparison["days_excluded_nonwear_or_suspicious"] = (
    day_comparison["total_days_minute_data"] - day_comparison["valid_days_minute_data"]
)
day_comparison["days_excluded_oai_cap"] = (
    day_comparison["valid_days_minute_data"] - day_comparison["oai_valid_days"]
)

print(f"Total participants: {len(day_comparison)}")
print(f"\nDay comparison statistics:")
print(day_comparison[[
    "total_days_minute_data",
    "valid_days_minute_data",
    "oai_valid_days",
    "valid_days_oai",
    "days_excluded_total",
    "days_excluded_nonwear_or_suspicious",
    "days_excluded_oai_cap",
]].describe())

print(f"\nParticipants with discrepancies:")
discrepant = day_comparison[
    day_comparison["valid_days_minute_data"] != day_comparison["oai_valid_days"]
].dropna()
print(f"Total discrepant: {len(discrepant)}")
print(discrepant.sort_values("days_excluded_total", ascending=False))

In [ ]:
# get day-level data for discrepant participants
discrepant_ids = discrepant["ID"].tolist()

# get all days from minute data for discrepant participants
discrepant_days = daily_activity_onset_offset_metrics_08[
    daily_activity_onset_offset_metrics_08["ID"].isin(discrepant_ids)
][["ID", "study_day", "wear_duration_min"]]

# add suspicious minutes per day
suspicious_per_day = (
    minute_metrics_08[minute_metrics_08["ID"].isin(discrepant_ids)]
    .groupby(["ID", "study_day"])["is_suspicious"]
    .sum()
    .reset_index()
    .rename(columns={"study_day": "study_day", "is_suspicious": "suspicious_minutes"})
)

discrepant_days = discrepant_days.merge(suspicious_per_day, on=["ID", "study_day"], how="left")

# flag days that are in minute data but NOT in daily_metrics_08 (excluded by OAI)
oai_days = daily_metrics_08[["ID", "study_day"]].drop_duplicates()
oai_days["in_oai_daily"] = True

discrepant_days = discrepant_days.merge(oai_days, on=["ID", "study_day"], how="left")
discrepant_days["in_oai_daily"] = discrepant_days["in_oai_daily"].fillna(False)

# only look at excluded days
excluded_days = discrepant_days[discrepant_days["in_oai_daily"] == False]

print(f"Total excluded days across discrepant participants: {len(excluded_days)}")
print(f"\nExclusion reasons:")
print(f"Days with suspicious minutes > 0: {(excluded_days['suspicious_minutes'] > 0).sum()} ({(excluded_days['suspicious_minutes'] > 0).mean():.1%})")
print(f"Days with wear duration < 600 min: {(excluded_days['wear_duration_min'] < 600).sum()} ({(excluded_days['wear_duration_min'] < 600).mean():.1%})")
print(f"Days with both suspicious AND < 600 min: {((excluded_days['suspicious_minutes'] > 0) & (excluded_days['wear_duration_min'] < 600)).sum()}")
print(f"Days with neither reason (OAI 7-day cap): {((excluded_days['suspicious_minutes'] == 0) & (excluded_days['wear_duration_min'] >= 600)).sum()}")

## 5.5 Harmonic regression

five features per participant.
Three from harmonic regression (MESOR, amplitude, acrophase) describing
the *shape* of the average daily rhythm. Two nonparametric indices
(IV, IS) describing within-day fragmentation and between-day consistency,
which the harmonic fit cannot capture by construction.

### 5.5.1 Mean daily curve

In [ ]:
# gropupby ID, minute_sequence

mean_daily_curve = (
    minute_metrics_08
    .groupby(["ID", "minute_sequence"])["counts"]
    .mean()
    .reset_index()
    .rename(columns={"counts": "mean_counts"})
)

print(f"Mean daily curve computed for {mean_daily_curve['ID'].nunique():,} participants.")
print(mean_daily_curve.head())

In [ ]:
# Plot mean daily curve for a few random participants to verify the shape (sanity check)

sample_ids = mean_daily_curve["ID"].drop_duplicates().sample(5, random_state=42)

fig, ax = plt.subplots(figsize=(14, 4))

for participant_id in sample_ids:
    participant_curve = mean_daily_curve[mean_daily_curve["ID"] == participant_id]
    ax.plot(
        participant_curve["minute_sequence"],
        participant_curve["mean_counts"],
        alpha=0.7,
        linewidth=0.8,
        label=str(participant_id),
    )

ax.set_xlabel("Minute sequence")
ax.set_ylabel("Mean counts")
ax.set_title("Mean daily activity curve — 5 random participants")
ax.set_xticks([0, 180, 360, 540, 720, 900, 1080, 1260, 1440])
ax.set_xticklabels(["00:00", "03:00", "06:00", "09:00", "12:00", "15:00", "18:00", "21:00", "24:00"])
ax.legend(title="ID", fontsize=8)
plt.tight_layout()
plt.show()

### 5.5.2 Single-participant demonstration

intercept (MESOR) plus two harmonic
pairs at periods of 24h and 12h, fitted with ordinary least squares.
Note that acrophase is read from argmax of the fitted curve rather
than from arctan2 of the coefficients to avoid sign-convention bugs.

In [ ]:
# manual fit for single participant

# Worked example on a single participant, selected by position so that no
# participant identifier appears in this file.
participant_id = mean_daily_curve["ID"].unique()[0]

single_curve = mean_daily_curve[mean_daily_curve["ID"] == participant_id]

minutes = single_curve["minute_sequence"].to_numpy(dtype=float)
counts = single_curve["mean_counts"].to_numpy(dtype=float)

# Build design matrix: intercept + 2 harmonic pairs
period = 1440
design_matrix = np.column_stack([
    np.ones(len(minutes)),                                    # MESOR
    np.cos(2 * np.pi * 1 * minutes / period),                # harmonic 1 cosine
    np.sin(2 * np.pi * 1 * minutes / period),                # harmonic 1 sine
    np.cos(2 * np.pi * 2 * minutes / period),                # harmonic 2 cosine
    np.sin(2 * np.pi * 2 * minutes / period),                # harmonic 2 sine
])

coefficients, _, _, _ = lstsq(design_matrix, counts, rcond=None)

mesor = coefficients[0]
amplitude = np.sqrt(coefficients[1]**2 + coefficients[2]**2)
# Read acrophase directly from the peak of the fitted curve
# rather than computing from arctan2 to avoid sign convention issues
fitted_counts = design_matrix @ coefficients
peak_minute = minutes[np.argmax(fitted_counts)]
acrophase_hours = peak_minute / 60

print(f"MESOR: {mesor:.2f}")
print(f"Amplitude: {amplitude:.2f}")
print(f"Acrophase: {acrophase_hours:.2f} hours ({int(peak_minute // 60):02d}:{int(peak_minute % 60):02d})")

In [ ]:
# plot showing curve, fit, MESOR line, acrophase line

fitted_counts = design_matrix @ coefficients

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(minutes, counts, alpha=0.4, linewidth=0.8, label="Mean curve")
ax.plot(minutes, fitted_counts, linewidth=2, color="red", label="Harmonic fit")
ax.axhline(y=mesor, color="green", linestyle="--", linewidth=1, label=f"MESOR ({mesor:.0f})")
ax.axvline(x=acrophase_hours * 60, color="orange", linestyle="--", linewidth=1, label=f"Acrophase ({acrophase_hours:.1f}h)")
ax.set_xlabel("Minute sequence")
ax.set_ylabel("Counts")
ax.set_title(f"Harmonic fit — participant {participant_id}")
ax.set_xticks([0, 180, 360, 540, 720, 900, 1080, 1260, 1440])
ax.set_xticklabels(["00:00", "03:00", "06:00", "09:00", "12:00", "15:00", "18:00", "21:00", "24:00"])
ax.legend()
plt.tight_layout()
plt.show()


### 5.5.3 Fit to all participants (MESOR, amplitude, acrophase)

In [ ]:
# fit_harmonic_model function

def fit_harmonic_model(
        mean_daily_curve: pd.DataFrame,
        column_id: str,
        column_minute_sequence: str,
        column_mean_counts: str,
        period: int = 1440,
        number_of_harmonics: int = 2,
) -> pd.DataFrame:

    # Fit a harmonic regression model to each participant's mean daily curve and extract MESOR, amplitude, and acrophase.

    records = []

    for participant_id, participant_curve in mean_daily_curve.groupby(column_id):
        minutes = participant_curve[column_minute_sequence].to_numpy(dtype=float)
        counts = participant_curve[column_mean_counts].to_numpy(dtype=float)

        # Build design matrix
        design_matrix = [np.ones(len(minutes))]
        for harmonic_index in range(1, number_of_harmonics + 1):
            design_matrix.append(np.cos(2 * np.pi * harmonic_index * minutes / period))
            design_matrix.append(np.sin(2 * np.pi * harmonic_index * minutes / period))
        design_matrix = np.column_stack(design_matrix)

        coefficients, _, _, _ = lstsq(design_matrix, counts, rcond=None)

        mesor = coefficients[0]
        amplitude = np.sqrt(coefficients[1]**2 + coefficients[2]**2)

        fitted_counts = design_matrix @ coefficients
        peak_minute = minutes[np.argmax(fitted_counts)]
        acrophase_hour = peak_minute /60

        records.append({
            column_id: participant_id,
            "mesor":mesor,
            "amplitude":amplitude,
            "acrophase":acrophase_hour,
        })

    return pd.DataFrame(records).set_index(column_id)

harmonic_features = fit_harmonic_model(
    mean_daily_curve = mean_daily_curve,
    column_id = "ID",
    column_minute_sequence = "minute_sequence",
    column_mean_counts = "mean_counts",
)

print(f"Harmonic features extracted for {len(harmonic_features)} participants:")
print(harmonic_features.describe())


### 5.5.4 Acrophase distribution / outlier inspection

In [ ]:
# find participants with acrophase < 6 or > 20

print(harmonic_features[harmonic_features["acrophase"] < 6])
print(harmonic_features[harmonic_features["acrophase"] > 20])

In [ ]:
# Plot two outlier examples

fig, axes = plt.subplots(nrows=2, ncols=1, figsize=(14, 8))

# Selected by position from the acrophase extremes so that no participant
# identifier appears in this file.
outlier_ids = [harmonic_features["acrophase"].idxmin(), harmonic_features["acrophase"].idxmax()]

for ax, participant_id in zip(axes, outlier_ids):
    participant_curve = mean_daily_curve[mean_daily_curve["ID"] == participant_id]
    minutes = participant_curve["minute_sequence"].to_numpy(dtype=float)
    counts = participant_curve["mean_counts"].to_numpy(dtype=float)

    design_matrix = np.column_stack([
        np.ones(len(minutes)),
        np.cos(2 * np.pi * 1 * minutes / 1440),
        np.sin(2 * np.pi * 1 * minutes / 1440),
        np.cos(2 * np.pi * 2 * minutes / 1440),
        np.sin(2 * np.pi * 2 * minutes / 1440),
    ])
    coefficients, _, _, _ = lstsq(design_matrix, counts, rcond=None)
    fitted_counts = design_matrix @ coefficients

    ax.plot(minutes, counts, alpha=0.4, linewidth=0.8, label="Mean curve")
    ax.plot(minutes, fitted_counts, linewidth=2, color="red", label="Harmonic fit")
    ax.set_title(f"Participant {participant_id} — acrophase {harmonic_features.loc[participant_id, 'acrophase']:.1f}h")
    ax.set_xticks([0, 180, 360, 540, 720, 900, 1080, 1260, 1440])
    ax.set_xticklabels(["00:00", "03:00", "06:00", "09:00", "12:00", "15:00", "18:00", "21:00", "24:00"])
    ax.legend()

plt.tight_layout()
plt.show()

## 5.6 Rhythm indices: Intradaily Variability (IV) and Interdaily Stability (IS)
IV and IS are computed from the **raw minute-level data across all valid days**,
not from the mean daily curve. This is intentional, both indices specifically
quantify variability over time, which the mean curve averages away by design.

### 5.6.1 IV / IS single-participant demonstration

In [ ]:
# Worked example on a single participant, selected by position so that no
# participant identifier appears in this file.
participant_id = minute_metrics_08["ID"].unique()[0]

participant_data = (
    minute_metrics_08[minute_metrics_08["ID"] == participant_id]
    .sort_values(["study_day", "minute_sequence"])
)

counts = participant_data["counts"].to_numpy(dtype=float)

# IV: ratio of mean squared first-order differences to overall variance
# n * sum of squared differences between consecutive minutes
# divided by (n-1) * overall variance
number_of_minutes = len(counts)
overall_mean = np.mean(counts)
overall_variance = np.sum((counts - overall_mean) ** 2)
squared_differences = np.sum(np.diff(counts) ** 2)

iv = (number_of_minutes * squared_differences) / ((number_of_minutes - 1) * overall_variance)

print(f"IV for participant {participant_id}: {iv:.4f}")

In [ ]:
# IS: ratio of variance of the mean 24h profile to overall variance
# Reshape counts into a matrix of days x minutes
number_of_complete_days = len(counts) // 1440
trimmed_counts = counts[:number_of_complete_days * 1440]
reshaped = trimmed_counts.reshape(number_of_complete_days, 1440)

# Mean activity at each of the 1440 minute positions across all days
mean_24h_profile = np.mean(reshaped, axis=0)
overall_mean = np.mean(trimmed_counts)

profile_variance = np.sum((mean_24h_profile - overall_mean) ** 2)
overall_variance = np.sum((trimmed_counts - overall_mean) ** 2)

is_index = (number_of_complete_days * profile_variance) / overall_variance

print(f"IS for participant {participant_id}: {is_index:.4f}")

### 5.6.2 Apply IV / IS to all participants

In [ ]:
# compute_iv_and_is function
def compute_iv_and_is(
    minute_dataframe: pd.DataFrame,
    column_id: str,
    column_study_day: str,
    column_minute_sequence: str,
    column_counts: str,
    minutes_per_day: int = 1440,
) -> pd.DataFrame:
    """
    Compute intradaily variability (IV) and interdaily stability (IS)
    for each participant from the raw minute-level activity data.

    IV and IS are computed from the raw multi-day signal rather than
    the mean daily curve because both indices specifically quantify
    variability over time, which the mean curve averages away.

    :param minute_dataframe: Cleaned minute-level DataFrame.
    :param column_id: Participant identifier column.
    :param column_study_day: Study day column.
    :param column_minute_sequence: Minute sequence column.
    :param column_counts: Activity counts column.
    :param minutes_per_day: Number of minutes per day (1440).
    :return: DataFrame indexed by participant ID with columns
        [intradaily_variability, interdaily_stability].
    """
    records = []

    for participant_id, participant_data in minute_dataframe.groupby(column_id):
        participant_data = participant_data.sort_values(
            by=[column_study_day, column_minute_sequence]
        )
         # Fill residual NaN values with 0 before computing variance-based indices
        counts = participant_data[column_counts].to_numpy(dtype=float)
        counts = np.nan_to_num(counts, nan=0.0)

        number_of_minutes = len(counts)
        overall_mean = np.mean(counts)
        overall_variance = np.sum((counts - overall_mean) ** 2)

        # IV
        squared_differences = np.sum(np.diff(counts) ** 2)
        iv = (number_of_minutes * squared_differences) / ((number_of_minutes - 1) * overall_variance)

        # IS
        number_of_complete_days = number_of_minutes // minutes_per_day
        trimmed_counts = counts[:number_of_complete_days * minutes_per_day]
        reshaped = trimmed_counts.reshape(number_of_complete_days, minutes_per_day)
        mean_24h_profile = np.mean(reshaped, axis=0)
        overall_mean_trimmed = np.mean(trimmed_counts)
        profile_variance = np.sum((mean_24h_profile - overall_mean_trimmed) ** 2)
        overall_variance_trimmed = np.sum((trimmed_counts - overall_mean_trimmed) ** 2)
        is_index = (number_of_complete_days * profile_variance) / overall_variance_trimmed

        records.append({
            column_id: participant_id,
            "intradaily_variability": iv,
            "interdaily_stability": is_index,
        })

    return pd.DataFrame(records).set_index(column_id)

# apply function
rhythm_indices = compute_iv_and_is(
    minute_dataframe=minute_metrics_08,
    column_id="ID",
    column_study_day="study_day",
    column_minute_sequence="minute_sequence",
    column_counts="counts",
)

# describe IV an IS
print(f"IV and IS computed for {len(rhythm_indices)} participants.")
print(rhythm_indices.describe())

### Merge features into the summary metrics_08

In [ ]:
summary_metrics_08 = summary_metrics_08.set_index("ID")

summary_metrics_08 = summary_metrics_08.join(harmonic_features, how="left")
summary_metrics_08 = summary_metrics_08.join(rhythm_indices, how="left")

print(f"Summary data shape after merging: {summary_metrics_08.shape}")
print(f"Missing values for new features:")
print(summary_metrics_08[["mesor", "amplitude", "acrophase",
                           "intradaily_variability", "interdaily_stability"]].isna().sum())

## 5.7 Per-day harmonic decomposition

In [ ]:
# define weekend days and weekday days

weekend_days = ["Saturday", "Sunday"]
daily_metrics_08["day_type"] = daily_metrics_08["week_day"].apply(
    lambda day: "weekend" if day in weekend_days else "weekday"
)

### 5.7.1 Single-day demonstration

In [ ]:
# Participant and study day selected by position so that no participant
# identifier appears in this file.
participant_id = minute_metrics_08["ID"].unique()[0]
study_day = minute_metrics_08.loc[minute_metrics_08["ID"] == participant_id, "study_day"].unique()[0]

single_day = (
    minute_metrics_08[
        (minute_metrics_08["ID"] == participant_id) &
        (minute_metrics_08["study_day"] == study_day)
    ]
    .sort_values("minute_sequence")
)

minutes = single_day["minute_sequence"].to_numpy(dtype=float)
counts = single_day["counts"].to_numpy(dtype=float)
counts = np.nan_to_num(counts, nan=0.0)

# Harmonic model
period = 1440
design_matrix = np.column_stack([
    np.ones(len(minutes)),
    np.cos(2 * np.pi * 1 * minutes / period),
    np.sin(2 * np.pi * 1 * minutes / period),
    np.cos(2 * np.pi * 2 * minutes / period),
    np.sin(2 * np.pi * 2 * minutes / period),
])

coefficients, _, _, _ = lstsq(design_matrix, counts, rcond=None)
fitted_counts = design_matrix @ coefficients

mesor = coefficients[0]
amplitude = np.sqrt(coefficients[1]**2 + coefficients[2]**2)
peak_minute = minutes[np.argmax(fitted_counts)]
acrophase_hours = peak_minute / 60

# IV
number_of_minutes = len(counts)
overall_mean = np.mean(counts)
overall_variance = np.sum((counts - overall_mean) ** 2)
squared_differences = np.sum(np.diff(counts) ** 2)
iv = (number_of_minutes * squared_differences) / ((number_of_minutes - 1) * overall_variance)

print(f"MESOR: {mesor:.2f}")
print(f"Amplitude: {amplitude:.2f}")
print(f"Acrophase: {acrophase_hours:.2f}h")
print(f"IV: {iv:.4f}")

### 5.7.2 Per-day harmonic + IV across all participant-days

same harmonic specification as section 2, but fitted to each participant-day independently rather than to the mean curve

In [ ]:
# harmonic regression features and iv function

def extract_daily_harmonic_and_iv(
    minute_dataframe: pd.DataFrame,
    column_id: str,
    column_study_day: str,
    column_minute_sequence: str,
    column_counts: str,
    period: int = 1440,
    number_of_harmonics: int = 2,
) -> pd.DataFrame:
    """
    Fit harmonic regression and compute intradaily variability (IV)
    for each participant-day.

    Unlike the participant-level harmonic features which are fitted to the
    mean daily curve, these features are computed per individual day to enable
    weekday vs weekend and employment status comparisons.

    IS is excluded here as it requires multiple days by definition and
    remains a participant-level feature only.

    :param minute_dataframe: Cleaned minute-level DataFrame.
    :param column_id: Participant identifier column.
    :param column_study_day: Study day column.
    :param column_minute_sequence: Minute sequence column.
    :param column_counts: Activity counts column.
    :param period: Period in minutes (1440 for daily rhythm).
    :param number_of_harmonics: Number of harmonic pairs to fit.
    :return: DataFrame with one row per participant-day containing
        mesor, amplitude, acrophase, and intradaily_variability.
    """
    records = []

    for (participant_id, study_day), day_data in minute_dataframe.groupby(
        [column_id, column_study_day]
    ):
        day_data = day_data.sort_values(column_minute_sequence)
        minutes = day_data[column_minute_sequence].to_numpy(dtype=float)
        counts = np.nan_to_num(
            day_data[column_counts].to_numpy(dtype=float), nan=0.0
        )

        # Build design matrix
        design_matrix = [np.ones(len(minutes))]
        for harmonic_index in range(1, number_of_harmonics + 1):
            design_matrix.append(np.cos(2 * np.pi * harmonic_index * minutes / period))
            design_matrix.append(np.sin(2 * np.pi * harmonic_index * minutes / period))
        design_matrix = np.column_stack(design_matrix)

        coefficients, _, _, _ = lstsq(design_matrix, counts, rcond=None)
        fitted_counts = design_matrix @ coefficients

        mesor = coefficients[0]
        amplitude = np.sqrt(coefficients[1]**2 + coefficients[2]**2)
        peak_minute = minutes[np.argmax(fitted_counts)]
        acrophase_hours = peak_minute / 60

        # IV
        number_of_minutes = len(counts)
        overall_mean = np.mean(counts)
        overall_variance = np.sum((counts - overall_mean) ** 2)
        squared_differences = np.sum(np.diff(counts) ** 2)
        iv = (
            (number_of_minutes * squared_differences)
            / ((number_of_minutes - 1) * overall_variance)
            if overall_variance > 0 else np.nan
        )

        records.append({
            column_id: participant_id,
            column_study_day: study_day,
            "mesor_daily": mesor,
            "amplitude_daily": amplitude,
            "acrophase_daily": acrophase_hours,
            "intradaily_variability_daily": iv,
        })

    return pd.DataFrame(records)


# apply function
daily_harmonic_features = extract_daily_harmonic_and_iv(
    minute_dataframe=minute_metrics_08,
    column_id="ID",
    column_study_day="study_day",
    column_minute_sequence="minute_sequence",
    column_counts="counts",
)

print(f"Daily harmonic features extracted for {len(daily_harmonic_features)} participant-days.")
print(daily_harmonic_features.describe())

### Merge features to daily_metrics_08 and save as CSV

In [ ]:

# merge daily harmonic features to daily_metrics_08

daily_metrics_08 = daily_metrics_08.merge(
    daily_harmonic_features[
        [
            "ID",
            "study_day",
            "mesor_daily",
            "amplitude_daily",
            "acrophase_daily",
            "intradaily_variability_daily",
        ]
    ],
    on=["ID", "study_day"],
    how="left",
)

daily_metrics_08.to_csv(output_path / "daily_metrics_08.csv", sep="|", index=False)

### 5.7.3 Day-type mean-curve harmonic (weekday / weekend)

In [ ]:
# function to extract main curve for weekends and weekdays
def extract_mean_curve_harmonic_by_day_type(
    minute_dataframe: pd.DataFrame,
    daily_metadata_dataframe: pd.DataFrame,
    column_id: str,
    column_study_day: str,
    column_minute_of_day: str,
    column_counts: str,
    column_day_type: str,
    period: int = 1440,
    number_of_harmonics: int = 2,
) -> pd.DataFrame:
    """
    Fit harmonic regression to the participant-level mean daily activity
    curve, separately for weekday and weekend days.

    For each participant and each day type, minute-level activity counts
    are averaged across days at every minute-of-day, producing a single
    24-hour mean curve. A harmonic regression is then fitted to that mean
    curve to derive mesor, amplitude, and acrophase.

    :param minute_dataframe: Cleaned minute-level DataFrame containing
        activity counts.
    :param daily_metadata_dataframe: Daily-level DataFrame containing the
        day type label (weekday / weekend) for each participant-day.
    :param column_id: Participant identifier column.
    :param column_study_day: Study day column.
    :param column_minute_of_day: Minute-of-day column (0 to 1439).
    :param column_counts: Activity counts column.
    :param column_day_type: Column labelling each day as weekday or weekend.
    :param period: Period in minutes (1440 for a daily rhythm).
    :param number_of_harmonics: Number of harmonic pairs to fit.
    :return: DataFrame with one row per participant containing mesor,
        amplitude, and acrophase for both weekday and weekend mean curves.
    """
    minute_with_day_type = minute_dataframe.merge(
        daily_metadata_dataframe[[column_id, column_study_day, column_day_type]].drop_duplicates(),
        on=[column_id, column_study_day],
        how="left",
    )

    records = []

    for (participant_id, day_type), participant_day_type_data in minute_with_day_type.groupby(
        [column_id, column_day_type]
    ):
        mean_curve = (
            participant_day_type_data
            .groupby(column_minute_of_day)[column_counts]
            .mean()
            .sort_index()
        )

        minutes = mean_curve.index.to_numpy(dtype=float)
        counts = np.nan_to_num(mean_curve.to_numpy(dtype=float), nan=0.0)

        design_matrix_columns = [np.ones(len(minutes))]
        for harmonic_index in range(1, number_of_harmonics + 1):
            design_matrix_columns.append(
                np.cos(2 * np.pi * harmonic_index * minutes / period)
            )
            design_matrix_columns.append(
                np.sin(2 * np.pi * harmonic_index * minutes / period)
            )
        design_matrix = np.column_stack(design_matrix_columns)

        coefficients, _, _, _ = lstsq(design_matrix, counts, rcond=None)
        fitted_counts = design_matrix @ coefficients

        mesor = coefficients[0]
        amplitude = np.sqrt(coefficients[1] ** 2 + coefficients[2] ** 2)
        peak_minute = minutes[np.argmax(fitted_counts)]
        acrophase_hours = peak_minute / 60

        records.append(
            {
                column_id: participant_id,
                column_day_type: day_type,
                "mesor_mean_curve": mesor,
                "amplitude_mean_curve": amplitude,
                "acrophase_mean_curve": acrophase_hours,
            }
        )

    long_format = pd.DataFrame(records)

    wide_format = long_format.pivot(
        index=column_id,
        columns=column_day_type,
        values=["mesor_mean_curve", "amplitude_mean_curve", "acrophase_mean_curve"],
    )
    wide_format.columns = [
        f"{feature_name}_{day_type_label}"
        for feature_name, day_type_label in wide_format.columns
    ]
    wide_format = wide_format.reset_index()

    return wide_format

In [ ]:
# merge curves into summery metrics
mean_curve_features_by_day_type = extract_mean_curve_harmonic_by_day_type(
    minute_dataframe=minute_metrics_08,
    daily_metadata_dataframe=daily_metrics_08,
    column_id="ID",
    column_study_day="study_day",
    column_minute_of_day="minute_sequence",   # adjust if your column is named differently
    column_counts="counts",
    column_day_type="day_type",
)

summary_metrics_08 = summary_metrics_08.merge(
    mean_curve_features_by_day_type,
    on="ID",
    how="left",
)

### 5.7.4 IV by day type

In [ ]:
iv_by_daytype = (
    daily_metrics_08.assign(
        day_type=lambda dataframe : dataframe["week_day"].
        isin(weekend_days).
        map({True: "weekend", False: "weekday"})
    )
    .groupby(["ID", "day_type"]) ["intradaily_variability_daily"]
    .mean().unstack("day_type")
   .rename(columns={
        "weekday": "iv_weekday",
        "weekend": "iv_weekend",
    })
             )

# Merge into the main summary feature dataframe
summary_metrics_08 = summary_metrics_08.merge(
    iv_by_daytype.reset_index(),
    on="ID",
    how="left",
)

## 5.8 Merge all features into summary_metrics_08

In [ ]:
# save to csv
summary_metrics_08.to_csv(output_path / "summary_metrics_08.csv", sep="|", index=False)

# 6. Participants description

In [ ]:
print(
    f"Participants after synchronisation:\n"
    f"  minute_metrics_08 : {minute_metrics_08['ID'].nunique():,}\n"
    f"  daily_metrics_08  : {daily_metrics_08['ID'].nunique():,}\n"
    f"  summary_metrics_08: {len(summary_metrics_08):,}"
)

In [ ]:
print("Missing values per outcome variable:")
total_rows = len(summary_metrics_08)
for column_name in final_outcome_variables:
    if column_name not in all_clinical_08_merged.columns:
        print(f"{column_name}: NOT PRESENT in dataframe")
        continue
    missing_count = summary_metrics_08[column_name].isna().sum()
    missing_share = missing_count / total_rows
    print(f"{column_name}: {missing_count}/{total_rows} ({missing_share:.1%})")

In [ ]:
def summarize_mean_and_standard_deviation(
    *,
    values: np.ndarray,
    decimal_places: int = 1,
) -> str:
    """Format a numeric array as ``mean (SD)`` using the sample standard deviation.

    Missing values encoded as ``np.nan`` are ignored in both statistics.

    :param values: One-dimensional array of observations, possibly containing ``np.nan``.
    :param decimal_places: Number of decimals to display for both statistics.
    :returns: A string of the form ``"12.3 (4.5)"``.
    """
    mean_value = np.nanmean(values)
    standard_deviation = np.nanstd(values, ddof=1)
    return f"{mean_value:.{decimal_places}f} ({standard_deviation:.{decimal_places}f})"


age_summary = summarize_mean_and_standard_deviation(
    values=summary_metrics_08["V08AGE"].to_numpy()
)
bmi_summary = summarize_mean_and_standard_deviation(
    values=summary_metrics_08["V08BMI"].to_numpy()
)

In [ ]:
def summarize_counts_and_percentages(
    *,
    values: pd.Series,
    category_order: list | None = None,
    percentage_base: str = "valid",
    decimal_places: int = 1,
) -> dict[object, str]:
    """Format a categorical column as ``n (%)`` per level for a descriptive table.

    :param values: Column of categorical observations, possibly containing ``NaN``.
    :param category_order: Explicit level ordering (for example ``[0, 1, 2, 3, 4]``
        for the KL grade). When ``None``, levels are ordered by descending count.
    :param percentage_base: Denominator for the percentage. Use ``"valid"`` to
        divide by the number of non-missing observations, or ``"total"`` to
        divide by the full column length including missing values.
    :param decimal_places: Number of decimals to display for the percentage.
    :returns: Mapping from each level to its ``"n (%)"`` string, in the resolved order.
    :raises ValueError: If ``percentage_base`` is not ``"valid"`` or ``"total"``.
    """
    if percentage_base == "valid":
        denominator = int(values.notna().sum())
    elif percentage_base == "total":
        denominator = int(len(values))
    else:
        raise ValueError(
            "percentage_base must be either 'valid' or 'total', "
            f"received {percentage_base!r}."
        )

    counts = values.value_counts(dropna=True)

    if category_order is not None:
        counts = counts.reindex(category_order, fill_value=0)

    summary = {}
    for level, count in counts.items():
        percentage = 100.0 * count / denominator if denominator else np.nan
        summary[level] = f"{int(count)} ({percentage:.{decimal_places}f})"
    return summary

In [ ]:
sex_summary = summarize_counts_and_percentages(
    values=summary_metrics_08["P02SEX"],
       category_order=["Male", "Female"],

)

kl_grade_summary = summarize_counts_and_percentages(
    values=summary_metrics_08["kl_grade_index_knee"],
    category_order=[0, 1, 2, 3, 4],
)

mvpa_guideline_summary = summarize_counts_and_percentages(
    values=summary_metrics_08["meets_weekly_who_guideline"],
    category_order=[True, False],
)

In [ ]:
def summarise_charlson_comorbidity(
    *,
    comorbidity_counts: pd.Series,
    two_or_more_label: str = "2 or more",
    decimal_places: int = 1,
) -> pd.DataFrame:
    """Summarise the modified Charlson comorbidity count for a baseline table.

    The right-skewed count is collapsed into the ordered categories ``"0"``,
    ``"1"`` and ``two_or_more_label``, then reported as count and percentage.
    Percentages are computed over participants with a non-missing value, so the
    categories sum to one hundred and the missing count is reported separately.

    :param comorbidity_counts: Integer comorbidity counts, one value per
        participant. Missing values must be genuine ``NaN`` rather than a coded
        sentinel.
    :type comorbidity_counts: pandas.Series
    :param two_or_more_label: Label used for counts of two or more.
    :type two_or_more_label: str
    :param decimal_places: Number of decimal places for the percentage.
    :type decimal_places: int
    :returns: One row per category plus a final ``"Missing"`` row, with count and
        formatted ``count (percentage)`` columns.
    :rtype: pandas.DataFrame
    """

    def assign_category(count_value: float) -> object:
        if pd.isna(count_value):
            return pd.NA
        if count_value <= 0:
            return "0"
        if count_value == 1:
            return "1"
        return two_or_more_label

    category_order = ["0", "1", two_or_more_label]
    categorised_values = comorbidity_counts.map(assign_category).astype("string")

    number_missing = int(categorised_values.isna().sum())
    number_non_missing = int(categorised_values.notna().sum())

    summary_rows: list[dict[str, object]] = []
    for category_label in category_order:
        count_for_category = int((categorised_values == category_label).sum())
        percentage_for_category = (
            100.0 * count_for_category / number_non_missing if number_non_missing else 0.0
        )
        summary_rows.append(
            {
                "Category": category_label,
                "Count": count_for_category,
                "Count (percentage)": (
                    f"{count_for_category} ({percentage_for_category:.{decimal_places}f})"
                ),
            }
        )

    summary_rows.append(
        {
            "Category": "Missing",
            "Count": number_missing,
            "Count (percentage)": str(number_missing),
        }
    )

    return pd.DataFrame(summary_rows)

In [ ]:
charlson_summary = summarise_charlson_comorbidity(
    comorbidity_counts=summary_metrics_08["V08COMORB"],
)

In [ ]:
baseline_rows = {
    "Age, years": age_summary,
    "BMI, kg/m²": bmi_summary,
    **{f"Sex: {level}": value for level, value in sex_summary.items()},
    **{f"KL grade {level}": value for level, value in kl_grade_summary.items()},
    **{f"MVPA guideline {level}": value for level, value in mvpa_guideline_summary.items()},
    **dict(zip(charlson_summary["Category"], charlson_summary["Count (percentage)"])),
}

baseline_table_v08 = pd.Series(baseline_rows).rename("Value").to_frame()
print(baseline_table_v08.to_string())

In [ ]:
zero_vigorous_proportion = (summary_metrics_08["mean_vigorous_bout_total_minutes"] == 0).mean()

print(zero_vigorous_proportion)

In [ ]:
grade_counts = pd.DataFrame(
    {
        "left": summary_metrics_08["V08XRKL_Left"].value_counts(dropna=False),
        "right": summary_metrics_08["V08XRKL_Right"].value_counts(dropna=False),
    }
).sort_index()

grade_percentage = (grade_counts / grade_counts.sum() * 100).round(1)

grade_distribution = grade_counts.join(
    grade_percentage,
    lsuffix="_count",
    rsuffix="_percent",
)

print(grade_distribution)

In [ ]:
print(summary_metrics_08[["V08KOOSKPR", "V08ICPTSKR", "V08KOOSKPL", "V08ICPTSKL"]].describe())

In [ ]:
OUTCOME_SCALE_BOUNDS = {
    # Pain
    "V08KOOSKPR": (0.0, 100.0),
    "V08KOOSKPL": (0.0, 100.0),
    "V08ICPTSKR": (0.0, 100.0),
    "V08ICPTSKL": (0.0, 100.0),
    # Performance function, no theoretical bounds
    "V0820MPACE": None,
    "V08CSTIME1": None,
    # Self-reported function and symptoms
    "V08WOMADLR": (0.0, 68.0),
    "V08WOMADLL": (0.0, 68.0),
    "V08KOOSYMR": (0.0, 100.0),
    "V08KOOSYML": (0.0, 100.0),
    # Participation
    "V08LLDIFST": (0.0, 100.0),
    "V08LLDILST": (0.0, 100.0),
    # Quality of life
    "V08KOOSQOL": (0.0, 100.0),
    # Depression
    "V08CESD": (0.0, 60.0),
    # KL grade
    "V08XRKL_Right": (0, 4),
    "V08XRKL_Left": (0, 4),
}

In [ ]:
SCALE_BOUND_TOLERANCE = 1e-9


def summarise_outcome_distributions(
    *,
    visit_data: pd.DataFrame,
    outcome_scale_bounds: dict[str, tuple[float, float] | None],
    decimal_places: int = 1,
) -> pd.DataFrame:
    """Describe the distribution of each outcome at a single visit.

    Returns the median with interquartile range, the observed range, and
    the proportion of participants sitting on the theoretical minimum and
    maximum of the instrument. Measures without a fixed scale range carry
    a dash in the two percentage columns.

    Values are counted as sitting on a bound when they reach or pass it
    within a small tolerance, so that rounding in a transformed score does
    not hide a genuine floor or ceiling concentration. Percentages are
    computed over the analysable values only, so missing entries are
    excluded from both the numerator and the denominator.

    :param visit_data: One row per participant, one column per outcome.
    :param outcome_scale_bounds: Mapping from column name to the
        theoretical ``(minimum, maximum)`` of the instrument, or ``None``
        for measures without a fixed scale range.
    :param decimal_places: Decimal places used in the formatted cells.
    :return: One row per outcome, ordered as in ``outcome_scale_bounds``.
    """
    summary_rows = []

    for outcome_name, scale_bounds in outcome_scale_bounds.items():
        observed_values = visit_data[outcome_name].dropna()
        if observed_values.empty:
            continue

        if scale_bounds is None:
            percentage_at_minimum = "—"
            percentage_at_maximum = "—"
        else:
            scale_minimum, scale_maximum = scale_bounds
            at_minimum = observed_values <= scale_minimum + SCALE_BOUND_TOLERANCE
            at_maximum = observed_values >= scale_maximum - SCALE_BOUND_TOLERANCE
            percentage_at_minimum = f"{100.0 * at_minimum.mean():.{decimal_places}f}"
            percentage_at_maximum = f"{100.0 * at_maximum.mean():.{decimal_places}f}"

        summary_rows.append(
            {
                "Outcome": outcome_name,
                "Median (IQR)": (
                    f"{observed_values.median():.{decimal_places}f} "
                    f"({observed_values.quantile(0.25):.{decimal_places}f}"
                    f"–{observed_values.quantile(0.75):.{decimal_places}f})"
                ),
                "Observed range": (
                    f"{observed_values.min():.{decimal_places}f}"
                    f"–{observed_values.max():.{decimal_places}f}"
                ),
                "% at scale minimum": percentage_at_minimum,
                "% at scale maximum": percentage_at_maximum,
            }
        )

    return pd.DataFrame(summary_rows)

In [ ]:
distribution_table = summarise_outcome_distributions(
    visit_data=summary_metrics_08,
    outcome_scale_bounds=OUTCOME_SCALE_BOUNDS,
)

print(distribution_table.to_markdown(index=False))